In [3]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
# Load Dataset
df = pd.read_csv("Telco-Customer-Churn.csv")

df.drop("customerID", axis=1, inplace=True)
# Numerical columns
num_cols = ["SeniorCitizen","tenure","MonthlyCharges"]
for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Categorical columns
cat_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod', 'TotalCharges']

for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)
df["Churn"] = df["Churn"].map({"Yes":1,"No":0})
    # Data Cleaning
print("\nMissing Values:")
print(df.isnull().sum())

df = df.drop_duplicates()

print("\nDuplicates Removed!")

num_cols = df.select_dtypes(include=['int64', 'float64']).columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

X = df.drop("Churn", axis=1)
y = df["Churn"]
X = pd.get_dummies(X)

encoded_cols = X.columns.to_list()

numeric_cols = ["SeniorCitizen","tenure","MonthlyCharges"]

Scaler = StandardScaler()
X[numeric_cols] = Scaler.fit_transform(X[numeric_cols])

# ===========================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)




Missing Values:
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

Duplicates Removed!


C:\Users\shaik\AppData\Local\Temp\ipykernel_19244\2188736058.py:15: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df[col].fillna(df[col].median(), inplace=True)
C:\Users\shaik\AppData\Local\Temp\ipykernel_19244\2188736058.py:15: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment

In [5]:
# ===========================================
# Q2: Classification Algorithms
# ===========================================

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import pandas as pd

# Dictionary of models
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Support Vector Machine": SVC(random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB()
}

# Store results
results = []

# ===========================================
# Train & Evaluate Models
# ===========================================

for name, model in models.items():

    print("="*60)
    print(name)
    print("="*60)

    # Train Model
    model.fit(X_train, y_train)

    # Prediction
    y_pred = model.predict(X_test)

    # Accuracy
    accuracy = accuracy_score(y_test, y_pred)

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)

    # Classification Report
    report = classification_report(y_test, y_pred)

    print(f"\nAccuracy : {accuracy:.4f}")

    print("\nConfusion Matrix")
    print(cm)

    print("\nClassification Report")
    print(report)

    # Save Results
    results.append({
        "Model": name,
        "Accuracy": accuracy
    })

# ===========================================
# Comparison Table
# ===========================================

comparison_df = pd.DataFrame(results)

comparison_df = comparison_df.sort_values(
    by="Accuracy",
    ascending=False
)

print("\n")
print("="*60)
print("MODEL COMPARISON")
print("="*60)

print(comparison_df)

# ===========================================
# Best Model
# ===========================================

best_model = comparison_df.iloc[0]

print("\nBest Classification Model")
print(best_model)

Logistic Regression

Accuracy : 0.7986

Confusion Matrix
[[930 123]
 [160 192]]

Classification Report
              precision    recall  f1-score   support

           0       0.85      0.88      0.87      1053
           1       0.61      0.55      0.58       352

    accuracy                           0.80      1405
   macro avg       0.73      0.71      0.72      1405
weighted avg       0.79      0.80      0.79      1405

Decision Tree

Accuracy : 0.7609

Confusion Matrix
[[902 151]
 [185 167]]

Classification Report
              precision    recall  f1-score   support

           0       0.83      0.86      0.84      1053
           1       0.53      0.47      0.50       352

    accuracy                           0.76      1405
   macro avg       0.68      0.67      0.67      1405
weighted avg       0.75      0.76      0.76      1405

Support Vector Machine

Accuracy : 0.7993

Confusion Matrix
[[948 105]
 [177 175]]

Classification Report
              precision    recall  f1-sc

In [6]:
# ===========================================
# Q4 : Best Model Selection & Saving
# ===========================================

import joblib

# -------------------------------------------------
# Best Classification Model
# -------------------------------------------------

classification_models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Support Vector Machine": SVC(random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB()
}

best_accuracy = 0
best_classifier = None
best_classifier_name = ""

for name, model in classification_models.items():

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)

    if acc > best_accuracy:
        best_accuracy = acc
        best_classifier = model
        best_classifier_name = name

print("="*50)
print("Best Classification Model")
print("="*50)
print(best_classifier_name)
print("Accuracy :", round(best_accuracy,4))


# -------------------------------------------------
# Save Classification Model
# -------------------------------------------------

joblib.dump(best_classifier, "best_classifier.pkl")
joblib.dump(Scaler, "classification_scaler.pkl")
joblib.dump(encoded_cols, "classification_columns.pkl")

print("\nClassification model saved successfully!")


Best Classification Model
Support Vector Machine
Accuracy : 0.7993

Classification model saved successfully!


In [7]:
import joblib

joblib.dump(models["Logistic Regression"], "logistic.pkl")
joblib.dump(models["Decision Tree"], "decision_tree.pkl")
joblib.dump(models["K-Nearest Neighbors"], "knn.pkl")
joblib.dump(models["Naive Bayes"], "naive_bayes.pkl")

['naive_bayes.pkl']